# MCP Demo — MAGIC v22 Orders & Complaints + Microsoft Learn

This notebook demonstrates how to connect an AI agent to **two MCP servers simultaneously** using a shared conversation session:

| MCP Server | URL | Auth |
|---|---|---|
| **Microsoft Learn** | `https://learn.microsoft.com/api/mcp` | None (public) |
| **MAGIC v22** | `http://localhost:9898/mcp` | API Key Bearer |

## Scenarios covered
1. **Azure Learn** — Ask a documentation question via the MS Learn MCP server
2. **Make an Order** — Place a new order for customer *Dileep Kumar*
3. **Get Customer Orders** — Retrieve all orders for Dileep Kumar
4. **Register a Complaint** — File a complaint against the newly created order
5. **Get Complaint Details** — Retrieve the complaint that was just registered
6. **Resolve Complaint** — Resolve the complaint via the Order Fulfillment team

> **Prerequisites**  
> - `magic-v22-mcp` server running at `http://localhost:9898/mcp`  
> - `az login` completed (for Azure OpenAI auth)  
> - Both `.env` files present (workspace root + `0-mcp-servers/magic-v22-mcp/.env`)

## 1. Imports

In [1]:
import os

import httpx
from dotenv import load_dotenv

from agent_framework import MCPStreamableHTTPTool
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

## 2. Configuration

Loads Azure credentials from the workspace root `.env` and the MAGIC v22 API key from the MCP server's own `.env`.

In [2]:
# Workspace root .env — Azure OpenAI credentials
load_dotenv(override=True)

# MAGIC v22 MCP server .env — API_KEY (does not overwrite already-loaded vars)
load_dotenv(dotenv_path="0-mcp-servers/magic-v22-mcp/.env", override=False)

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")
api_key = os.getenv("API_KEY")

print("Project Endpoint :", project_endpoint)
print("Model            :", model)
print("MAGIC API Key    :", "✓ loaded" if api_key else "✗ not found — check 0-mcp-servers/magic-v22-mcp/.env")

Project Endpoint : https://ramkumar-ms-foundry.services.ai.azure.com/api/projects/ramkumar-msf-project
Model            : gpt-4o
MAGIC API Key    : ✓ loaded


## 3. MCP Tools Setup

- **MS Learn** — public, no auth required  
- **MAGIC v22** — protected by API Key; passed as a `Bearer` header via a custom `httpx.AsyncClient`

In [3]:
# ── Microsoft Learn MCP (public, no auth) ─────────────────────────────────────
ms_learn_tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP Tool",
    url="https://learn.microsoft.com/api/mcp",
)

# ── MAGIC v22 MCP (API Key bearer auth) ───────────────────────────────────────
# MCPStreamableHTTPTool accepts a custom httpx.AsyncClient so we can inject
# the Authorization header for every outbound request to our protected server.
magic_v22_tool = MCPStreamableHTTPTool(
    name="MAGIC v22 Orders and Complaints Tool",
    url="http://localhost:9898/mcp",
    http_client=httpx.AsyncClient(
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=30.0,
    ),
)

print("✓ MS Learn MCP tool ready")
print("✓ MAGIC v22 MCP tool ready")

✓ MS Learn MCP tool ready
✓ MAGIC v22 MCP tool ready


## 4. Agent & Session Setup

A **single shared session** is created so the agent remembers context across all scenarios — e.g. the order ID created in Scenario 2 is automatically available when registering a complaint in Scenario 4.

In [4]:
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model,
    credential=credential,
)

agent = client.as_agent(
    name="OrdersComplaintsAgent",
    instructions=(
        "You are a helpful assistant that manages customer orders and complaints "
        "using the MAGIC v22 MCP server. You can also look up Microsoft documentation "
        "using the Microsoft Learn MCP tool. "
        "When performing order or complaint operations, always confirm the action taken "
        "and include the relevant IDs (order_id, complaint_id) in your response so they "
        "can be referenced in follow-up requests."
    ),
    tools=[ms_learn_tool, magic_v22_tool],
)

# Single session shared across all scenario cells
session = agent.create_session()

print(f"✓ Agent '{agent.name}' ready")

✓ Agent 'OrdersComplaintsAgent' ready


---
## Scenario 1 — Microsoft Learn MCP

Ask a documentation question that is answered using the MS Learn MCP server.

In [5]:
query = "How do I create an Azure Blob Storage container using the Azure CLI?"

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: How do I create an Azure Blob Storage container using the Azure CLI?

Agent:
Here’s how to create an Azure Blob Storage container using the Azure CLI:

### Single Container Creation
```bash
az storage container create \
    --name <your-container-name> \
    --account-name <your-storage-account-name> \
    --auth-mode login
```
Replace `<your-container-name>` with the name of the container you want to create and `<your-storage-account-name>` with your storage account name.

### Example with Account Access Key
```bash
az storage container create \
    --name sample-container \
    --account-name <storage-account-name> \
    --account-key <account-key>
```
Here, `<account-key>` represents the key of the storage account.

For additional options and complete documentation, refer to the [Azure Documentation](https://learn.microsoft.com/azure/storage/blobs/blob-containers-cli#create-a-container). 

Do you want me to help you automate this process?


---
## Scenario 2 — Make an Order

Place a new order for customer **Dileep Kumar** using the MAGIC v22 `make_order` tool.  
The agent will generate a product SKU, units, and order amount.

In [6]:
query = (
    "Please create a new order for customer 'Dileep Kumar'. "
    "Use product SKU 'LAPTOP-PRO-X1', 2 units at $1499.99 each. "
    "Add a remark: 'Urgent delivery requested'. "
    "Please confirm the order details including the order_id and order_number."
)

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: Please create a new order for customer 'Dileep Kumar'. Use product SKU 'LAPTOP-PRO-X1', 2 units at $1499.99 each. Add a remark: 'Urgent delivery requested'. Please confirm the order details including the order_id and order_number.

Agent:
The order has been successfully created for customer **Dileep Kumar**. Here are the details:

- **Order ID**: 12  
- **Order Number**: ORD10012  
- **Product SKU**: LAPTOP-PRO-X1  
- **Units Ordered**: 2  
- **Total Amount**: $1499.99 per unit (Total: $2999.98)  
- **Remarks**: Urgent delivery requested  
- **Status**: PENDING  

Let me know if further assistance is needed with this order.


---
## Scenario 3 — Get Orders of the Customer

Retrieve all orders placed by **Dileep Kumar** to confirm the order just created.

In [7]:
query = "Show me all orders for customer 'Dileep Kumar'."

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: Show me all orders for customer 'Dileep Kumar'.

Agent:
Here are all the orders for customer **Dileep Kumar**:

1. **Order ID**: 12  
   - **Order Number**: ORD10012  
   - **Product SKU**: LAPTOP-PRO-X1  
   - **Units Ordered**: 2  
   - **Total Amount**: $2999.98  
   - **Remarks**: Urgent delivery requested  
   - **Status**: PENDING  

Let me know if you'd like more details or assistance with these orders.


---
## Scenario 4 — Register a Complaint

File a complaint against the order created in Scenario 2.  
The agent uses the `order_id` already in the session context.

In [8]:
query = (
    "I would like to register a complaint for the order we just created for Dileep Kumar. "
    "The complaint is: 'The laptop arrived with a cracked screen and the keyboard is unresponsive.' "
    "This is a HIGH priority issue. Register it on behalf of 'Dileep Kumar'. "
    "Please share the complaint_id once registered."
)

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: I would like to register a complaint for the order we just created for Dileep Kumar. The complaint is: 'The laptop arrived with a cracked screen and the keyboard is unresponsive.' This is a HIGH priority issue. Register it on behalf of 'Dileep Kumar'. Please share the complaint_id once registered.

Agent:
The complaint has been successfully registered for **Dileep Kumar**. Here are the details:

- **Complaint ID**: 7  
- **Order ID**: 12  
- **Description**: The laptop arrived with a cracked screen and the keyboard is unresponsive.  
- **Priority**: HIGH  
- **Status**: OPEN  

Let me know if further action is needed on this complaint.


---
## Scenario 5 — Get Complaint Details

Fetch the full details of the complaint just registered to verify it was stored correctly.

In [9]:
query = "Get the full details of the complaint we just registered."

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: Get the full details of the complaint we just registered.

Agent:
Here are the full details of the registered complaint:

- **Complaint ID**: 7  
- **Complaint Date**: 2026-05-07  
- **Order ID**: 12  
- **Registered By**: Dileep Kumar  
- **Description**: The laptop arrived with a cracked screen and the keyboard is unresponsive.  
- **Priority**: HIGH  
- **Status**: OPEN  
- **Resolved By**: Not yet assigned  
- **Resolution Remarks**: None  

Let me know if you need assistance with resolving or escalating this complaint.


---
## Scenario 6 — Resolve the Complaint

Resolve the complaint using the **Order Fulfillment** team.  
Valid resolver teams: `ORDER_FULFILLMENT`, `CUSTOMER_SUPPORT`, `LOGISTICS`, `BILLING`, `RETURNS_AND_REFUNDS`, `QUALITY_ASSURANCE`, `TECHNICAL_SUPPORT`

In [10]:
query = (
    "Please resolve the complaint we just registered. "
    "It should be resolved by the 'ORDER_FULFILLMENT' team. "
    "Use this resolution remark: "
    "'A replacement unit has been dispatched via express delivery. "
    "Customer will receive it within 2 business days. Apologies for the inconvenience.' "
    "Confirm the updated complaint status after resolving."
)

print(f"User: {query}\n")
response = await agent.run(query, session=session)
print(f"Agent:\n{response}")

User: Please resolve the complaint we just registered. It should be resolved by the 'ORDER_FULFILLMENT' team. Use this resolution remark: 'A replacement unit has been dispatched via express delivery. Customer will receive it within 2 business days. Apologies for the inconvenience.' Confirm the updated complaint status after resolving.

Agent:
The complaint has been resolved by the **ORDER_FULFILLMENT** team. Here are the updated details:

- **Complaint ID**: 7  
- **Status**: RESOLVED  
- **Resolution Remarks**: A replacement unit has been dispatched via express delivery. Customer will receive it within 2 business days. Apologies for the inconvenience.  

Let me know if you need assistance with closing the complaint or further actions.
